### Part 0: Installing libraries

In [1]:
%pip install spacy

In [2]:
%pip install spacy-universal-sentence-encoder

  Preparing metadata (setup.py) ... done
  Created wheel for spacy-universal-sentence-encoder: filename=spacy_universal_sentence_encoder-0.4.6-py3-none-any.whl size=16541 sha256=be5afcbc41b2e64bc037031fbfd6fe760a5feb4ec0e82ba0bae727b4c16dc798
  Stored in directory: /root/.cache/pip/wheels/b8/c5/91/dc7e03329421578e554c8d2635f92fd528531123cce102c7f2
Successfully built spacy-universal-sentence-encoder


In [3]:
%pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.1 MB/s eta 0:00:00


In [4]:
%pip install geopy

### Part 1: Constructing the similarity index

In order to construct the similarity index, we use spacy, in particular a universal encoder which allows for a more accurate word embedding process. This will help us in detecting true similarity, and getting rid of false-similar sentences, such as sentences that include a large amount of stop-words (e.g sentences that use "the", "a", "he", "it", etc. a lot will not appear as similar, but rather the true meaning of the sentence will be analysed).

In [5]:
import spacy
import numpy as np
import pandas as pd
import networkx as nx
import spacy_universal_sentence_encoder
from tqdm import tqdm
from google.colab import files
from pyvis.network import Network
from IPython.display import display, HTML
import folium
from geopy.geocoders import Nominatim

In [6]:
nlp = spacy_universal_sentence_encoder.load_model('en_use_lg')

Downloaded https://tfhub.dev/google/universal-sentence-encoder-large/5, Total size: 577.10MB



We read the data from the reporting frameworks table that we are working on.

In [7]:
reporting_data = pd.read_excel('ReportingFrameworks.xlsx')

In [8]:
reporting_data

,Framework,Topic,Reference,Recommendation,G_Ref,Scope,Guidance
0,TCFD,Governance,TCFD Governance A,Describe the board’s oversight of climate-rela...,TCFD Governance A1,Generic,Consider including a discussion of the followi...
1,TCFD,Governance,TCFD Governance B,Describe management’s role in assessing and ma...,TCFD Governance B1,Generic,Consider including the following information:\...
2,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A1,Generic,Organisations should provide the following inf...
3,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A4,Generic,Organizations should consider providing a desc...
4,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A5,Banks,Banks should describe significant concentratio...
...,...,...,...,...,...,...,...
550,ESRS E4,MetricsandTargets,ESRS E4 Metrics and Targets 41,If the undertaking identified material impacts...,AR33: With regard to metrics on the extent and...,NaN,NaN
551,ESRS E4,MetricsandTargets,ESRS E4 Metrics and Targets 42,The undertaking shall disclose its anticipated...,AR39: The undertaking may include an assessmen...,NaN,NaN
552,ESRS E4,MetricsandTargets,ESRS E4 Metrics and Targets 43,The information required by paragraph 42 is in...,NaN,NaN,NaN
553,ESRS E4,MetricsandTargets,ESRS E4 Metrics and Targets 44,The objective of this Disclosure Requirement i...,NaN,NaN,NaN


In [9]:
def similarity_finder(data, columns, run_aggregation = True):
  """
  This is a function that is calculating the similarity index between all of the entries in a specific column, using the spacy framework.

  Parameters:
  "data" represents the data that we are putting in the format provided to us by Lloyd.
  "columns" contains, in order:
  - the column which contains the framework
  - the column which contains the topic
  - the column which contains the exact reference
  - the column which contains the entry that we want to compare (which will initially be the recommendation, but may become the guidance)

  Outputs:
  The output will be similar to the similarrequrements.csv file. This output will then be fed into a download function, in case we want to download the file,
  and a networkx code which will create a network of all of these links.
  """
  framework_column = columns[0]
  topic_column = columns[1]
  reference_column = columns[2]
  comparison_column = columns[3]

  if(run_aggregation):
    data = data.replace("ESRS E1", "ESRS")
    data = data.replace("ESRS E4", "ESRS")
    data = data.replace("IFRS S1", "IFRS")
    data = data.replace("IFRS S2", "IFRS")

  n = int(len(data)*(len(data)-1)/2)
  similarity_list = []

  # Create a progress bar for the total number of comparisons, while also creating the similarity table file
  with tqdm(total=n, desc="Comparing pairs") as pbar:
    for i in range(len(data)):
      curr_framework = data.iloc[i][framework_column]
      curr_topic = data.iloc[i][topic_column]
      curr_reference = data.iloc[i][reference_column]
      curr_comparison = data.iloc[i][comparison_column]
      curr_nlp = nlp(curr_comparison)
      for j in range(i+1, len(data)):
        next_framework = data.iloc[j][framework_column]
        next_topic = data.iloc[j][topic_column]
        next_reference = data.iloc[j][reference_column]
        next_comparison = data.iloc[j][comparison_column]
        next_nlp = nlp(next_comparison)
        if (next_framework != curr_framework) and (next_topic == curr_topic): #We are getting rid of the similarities within the same framework, which are bound to be high, and we only calculate similarities where they are useful, in the same topic
          similarity_score = curr_nlp.similarity(next_nlp)
          similarity_list.append({"Topic":curr_topic, "Framework 1": curr_framework, "Reference 1": curr_reference, "Recommendation 1": curr_comparison, "Framework 2": next_framework, "Reference 2": next_reference, "Recommendation 2": next_comparison, "Similarity": similarity_score})
        pbar.update(1)  # Update the progress bar after each comparison

  similarity_table = pd.DataFrame(similarity_list)
  similarity_table.drop_duplicates(inplace=True)
  similarity_table.reset_index(drop=True, inplace=True)
  return similarity_table

In [15]:
similarity_table = similarity_finder(reporting_data, ["Framework", "Topic", "Reference", "Recommendation"])

Comparing pairs: 100%|██████████| 153735/153735 [01:10<00:00, 2175.40it/s]


In [11]:
def download_table(table, filename):
  """
  Downloads tables as a CSV file in Google Colab.

  Parameters:
  - table: pandas DataFrame to download
  - filename: name of the file to save (without extension)

  """
  # Save the DataFrame to a CSV file
  table.to_csv(filename, index=False)

  # Download the file
  files.download(filename)

  print(f"File '{filename}' has been downloaded!")

In [ ]:
download_table(similarity_table, "similarity_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_table.csv' has been downloaded!


### Part 2: Constructing the network of similarity

Below, what we will be creating is a similarity metric between different frameworks. The list of topics from which one can choose below is as follows: Disclosure, Governance, Metrics and Targets, Risk Management and Strategy.

In [12]:
def similarity_network_table(original_list = ["Disclosure", "Governance", "MetricsandTargets", "RiskManagement", "Strategy"], original_data=similarity_table):
  """
  This is a function that is calculating the data required to build a network of the similarities of different frameworks, based on a list of topics.

  Parameters:
  "list" is the list of topics on which we want to base the similarity. It can contain all topics, but it must include at least one. It's originally defined as the list of all topics.
  "data" is the underlying data, which in this file should be the similarity table found before. It's originally defined as the similarity_table. We are assuming that the data is in the format of the similarity table.

  Outputs:
  A table containing the similarity between all of the frameworks, based on the list of topics. This similarity is calculated as the average of the similarities of all recommendations
  in the chosen topics.
  """
  filtered_data = original_data.copy()
  for i in range(len(filtered_data)):
    if not(filtered_data["Topic"][i] in original_list):
      filtered_data.drop(i, inplace=True)
  filtered_data.reset_index(drop=True, inplace=True)
  framework_1 = pd.unique(filtered_data["Framework 1"])
  framework_2 = pd.unique(filtered_data["Framework 2"])
  all_frameworks = np.concatenate((framework_1, framework_2))
  all_frameworks = pd.unique(all_frameworks)
  similarity_list = []
  with tqdm(total=len(all_frameworks)*(len(all_frameworks)-1)/2, desc="Comparing pairs") as pbar:
    for i in range(len(all_frameworks)):
      curr_framework = all_frameworks[i]
      for j in range(i+1,len(all_frameworks)):
        next_framework = all_frameworks[j]
        similarity = filtered_data.loc[(filtered_data["Framework 1"] == curr_framework) & (filtered_data["Framework 2"] == next_framework)]["Similarity"].mean()
        similarity_list.append({"Framework 1": curr_framework, "Framework 2": next_framework, "Similarity": similarity})
        pbar.update(1)

  similarity_network = pd.DataFrame(similarity_list)
  similarity_network.fillna(0, inplace=True)
  similarity_network.drop_duplicates(inplace=True)
  similarity_network.reset_index(drop=True, inplace=True)
  return similarity_network

In [13]:
similarity_network_table_all_metrics = similarity_network_table()
similarity_network_table_disclosure = similarity_network_table(["Disclosure"])
similarity_network_table_governance = similarity_network_table(["Governance"])
similarity_network_table_metrics = similarity_network_table(["MetricsandTargets"])
similarity_network_table_risk = similarity_network_table(["RiskManagement"])
similarity_network_table_strategy = similarity_network_table(["Strategy"])

Comparing pairs: 100%|██████████| 28/28.0 [00:00<00:00, 928.66it/s]
Comparing pairs: 100%|██████████| 3/3.0 [00:00<00:00, 779.61it/s]
Comparing pairs: 100%|██████████| 21/21.0 [00:00<00:00, 1443.09it/s]
Comparing pairs: 0it [00:00, ?it/s]
Comparing pairs: 0it [00:00, ?it/s]
Comparing pairs: 100%|██████████| 15/15.0 [00:00<00:00, 1060.38it/s]


In [16]:
similarity_table

,Topic,Framework 1,Reference 1,Recommendation 1,Framework 2,Reference 2,Recommendation 2,Similarity
0,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance A,Describe the board’s oversight of naturerelate...,0.861571
1,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance B,Describe management’s role in assessing and ma...,0.669910
2,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance C,Describe the organisation’s human rights polic...,0.429584
3,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,IFRS,IFRS S1 26,The objective of sustainability-related financ...,0.268225
4,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,IFRS,IFRS S1 27,"To achieve this objective, an entity shall dis...",0.210691
...,...,...,...,...,...,...,...,...
4797,Disclosure,MAS,MAS Disclosure 7.3,The insurer should review its disclosure regul...,ESRS,ESRS E1 Disclosure 113,The undertaking shall include in its sustainab...,0.130172
4798,Disclosure,MAS,MAS Disclosure 7.3,The insurer should review its disclosure regul...,ESRS,ESRS E1 Disclosure 114,When the undertaking includes in its sustainab...,0.211078
4799,Disclosure,MAS,MAS Disclosure 7.3,The insurer should review its disclosure regul...,ESRS,ESRS E1 Disclosure 115,The undertaking shall structure its sustainabi...,0.136216
4800,Disclosure,MAS,MAS Disclosure 7.3,The insurer should review its disclosure regul...,ESRS,ESRS E1 Disclosure 116,The disclosures required by sector-specific ES...,0.261648


We are now going to asign to each framework a country where it either originated from or is in use.

In [17]:
download_table(similarity_network_table_all_metrics, "similarity_network_table_all_metrics.csv")
download_table(similarity_network_table_disclosure, "similarity_network_table_disclosure.csv")
download_table(similarity_network_table_governance, "similarity_network_table_governance.csv")
download_table(similarity_network_table_metrics, "similarity_network_table_metrics.csv")
download_table(similarity_network_table_risk, "similarity_network_table_risk.csv")
download_table(similarity_network_table_strategy, "similarity_network_table_strategy.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_network_table_all_metrics.csv' has been downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

File 'similarity_network_table_risk.csv' has been downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_network_table_strategy.csv' has been downloaded!


### Map of regulations and adopting countries

In [ ]:
adoption_dict = {
    "Frameworks":["TCFD", "TNFD", "PRA", "IFRS", "TPT", "BMA", "MAS", "ESRS"],
    "Countries":[["Canada", "France", "Germany", "Italy", "Japan", "United Kingdom", "USA", "New Zealand", "Switzerland", "Singapore", "Brazil", "China", "South Africa"],
                 ["Brazil", "China", "Colombia", "Costa Rica", "Egypt", "India", "Indonesia", "Kenya", "Malaysia", "Mexico", "Morocco", "Nigeria", "Peru", "Philippines", "South Africa"],
                 ["United Kingdom"],
                 ["Turkey", "Bangladesh", "Brazil", "Australia", "Japan", "United Kingdom", "Canada", "Singapore", "New Zealand", "Nigeria", "South Africa", "Malaysia", "China"],
                 ["United Kingdom"],
                 ["Bermuda"],
                 ["Singapore"],
                 ["Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"]]
}